# 01. Data Understanding & Telemetry Ingestion Contract

This notebook performs comprehensive ingestion validation, schema inspection, operating hour (SMR) tracking, and ground-truth event mapping across the 4 continuous mining dozers.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from src.data.load_data import load_workbook
from src.utils.io import load_config

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

config = load_config()
workbook = load_workbook(config["project"]["raw_file"])

telemetry = workbook["telemetry"]
events = workbook["events"]
metadata = workbook["metadata"]
thresholds = workbook["thresholds"]


## 1. Dataset Overview & Schema Inspection


In [ ]:
print("=== TELEMETRY DATASET INFO ===")
telemetry.info()
print("\nTelemetry Shape:", telemetry.shape)
display(telemetry.head(10))


## 2. Fleet Breakdown & Temporal Continuity


In [ ]:
print("Observations per Dozer:")
print(telemetry["Machine ID"].value_counts())

fleet_summary = telemetry.groupby("Machine ID").agg(
    start_timestamp=("Timestamp", "min"),
    end_timestamp=("Timestamp", "max"),
    total_operating_hours=("Timestamp", "count"),
    min_smr=("SMR", "min"),
    max_smr=("SMR", "max")
).reset_index()

print("\n=== FLEET OPERATIONAL SUMMARY ===")
display(fleet_summary)


## 3. Ground-Truth Event Categorization

Only **Unplanned Failure** represents true mechanical failures. **False Alarm** and **Scheduled Maintenance** must be mapped to negative targets to avoid false-label corruption.


In [ ]:
print("=== GROUND TRUTH EVENTS ===")
display(events)

plt.figure(figsize=(8, 4))
sns.countplot(data=events, x="Category", palette="Set2")
plt.title("Event Breakdown by Category", fontsize=14)
plt.xlabel("Ground Truth Event Category")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


## 4. OEM Engineering Thresholds

OEM thresholds provide domain engineering guidance for feature extraction. They are features, **never labels**.


In [ ]:
print("=== OEM DOMAIN THRESHOLDS ===")
display(thresholds)


## 5. Telemetry Missing Value Audit


In [ ]:
missing_df = telemetry.isnull().sum().reset_index()
missing_df.columns = ["Column", "Missing_Count"]
missing_df["Missing_Percentage"] = (missing_df["Missing_Count"] / len(telemetry)) * 100
display(missing_df)
